# **DATA CLEANING**

**Pipeline**: Raw CSV → **Cleaning** → Feature Engineering → EDA → Analytics

## **1. Load & clean raw data**

In [68]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split

In [69]:
# ---------- Load raw files ----------
#read data
sales = pd.read_csv("Dataset/Raw_data/Online_Sales.csv")
customers = pd.read_excel("Dataset/Raw_data/CustomersData.xlsx")
tax = pd.read_excel("Dataset/Raw_data/Tax_amount.xlsx")
discount = pd.read_csv("Dataset/Raw_data/Discount_Coupon.csv")
marketing = pd.read_csv("Dataset/Raw_data/Marketing_Spend.csv")
print("Raw shapes:")
for name, df in [("sales",sales),("customers",customers),("tax",tax),("discount",discount),("marketing",marketing)]:
    print(f"  {name}: {df.shape}")

Raw shapes:
  sales: (52924, 10)
  customers: (1468, 4)
  tax: (20, 2)
  discount: (204, 4)
  marketing: (365, 3)


In [70]:
#data audit
datasets = {
    "sales": sales,
    "customers": customers,
    "tax": tax,
    "discount": discount,
    "marketing": marketing
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("="*40)
    print(df.shape)
    print(df.dtypes)
    print(df.isnull().sum())
    print(df.duplicated().sum())


SALES
(52924, 10)
CustomerID               int64
Transaction_ID           int64
Transaction_Date           str
Product_SKU                str
Product_Description        str
Product_Category           str
Quantity                 int64
Avg_Price              float64
Delivery_Charges       float64
Coupon_Status              str
dtype: object
CustomerID             0
Transaction_ID         0
Transaction_Date       0
Product_SKU            0
Product_Description    0
Product_Category       0
Quantity               0
Avg_Price              0
Delivery_Charges       0
Coupon_Status          0
dtype: int64
0

CUSTOMERS
(1468, 4)
CustomerID       int64
Gender             str
Location           str
Tenure_Months    int64
dtype: object
CustomerID       0
Gender           0
Location         0
Tenure_Months    0
dtype: int64
0

TAX
(20, 2)
Product_Category        str
GST                 float64
dtype: object
Product_Category    0
GST                 0
dtype: int64
0

DISCOUNT
(204, 4)
Month        

1. Sales

In [71]:
# =============================================
# CLEANING — SALES
# =============================================
sales["CustomerID"]    = sales["CustomerID"].astype(str)
sales["Transaction_ID"] = sales["Transaction_ID"].astype(str)
sales["Transaction_Date"] = pd.to_datetime(sales["Transaction_Date"], errors="coerce")

for col in ["Product_SKU", "Product_Description", "Product_Category", "Coupon_Status"]:
    sales[col] = sales[col].astype(str).str.strip()

for col in ["Quantity", "Avg_Price", "Delivery_Charges"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

sales = sales.drop_duplicates(subset=["CustomerID", "Transaction_ID", "Product_SKU"])
sales = sales[(sales["Quantity"] > 0) & (sales["Avg_Price"] > 0) & (sales["Delivery_Charges"] >= 0)]
sales = sales.dropna(subset=["CustomerID", "Transaction_ID", "Transaction_Date", "Product_SKU"])

sales["Product_Description"] = sales["Product_Description"].fillna("Unknown")
sales["Coupon_Status"]       = sales["Coupon_Status"].fillna("Not Used").str.title()

# Base revenue (trước tax/discount)
sales["Revenue"] = sales["Quantity"] * sales["Avg_Price"]
sales["Month"]   = sales["Transaction_Date"].dt.strftime("%b")

print(f"Sales clean: {sales.shape}")

Sales clean: (52924, 12)


In [72]:
# =============================================
# CLEANING — CUSTOMERS
# =============================================
customers["CustomerID"] = customers["CustomerID"].astype(str)

for col in ["Gender", "Location"]:
    customers[col] = customers[col].astype(str).str.strip().str.title()

customers = customers[customers["Tenure_Months"] >= 0]
print(f"Customers clean: {customers.shape}")

Customers clean: (1468, 4)


3. Discount

In [73]:
# =============================================
# CLEANING — DISCOUNT
# =============================================
for col in ["Month", "Product_Category", "Coupon_Code"]:
    discount[col] = discount[col].astype(str).str.strip()

discount["Discount_pct"] = pd.to_numeric(discount["Discount_pct"], errors="coerce").fillna(0)
# Normalize: nếu lưu dạng 10 thì / 100 để ra 0.10
if discount["Discount_pct"].max() > 1:
    discount["Discount_pct"] = discount["Discount_pct"] / 100

discount = discount[(discount["Discount_pct"] >= 0) & (discount["Discount_pct"] <= 1)]
print(f"Discount clean: {discount.shape}")

Discount clean: (204, 4)


4. Marketing

In [74]:
# =============================================
# CLEANING — MARKETING
# =============================================
marketing["Date"] = pd.to_datetime(marketing["Date"], errors="coerce")
marketing = marketing[(marketing["Offline_Spend"] >= 0) & (marketing["Online_Spend"] >= 0)]
marketing["Total_Marketing_Spend"] = marketing["Offline_Spend"] + marketing["Online_Spend"]
print(f"Marketing clean: {marketing.shape}")

Marketing clean: (365, 4)


5. Tax

In [75]:
# =============================================
# CLEANING — TAX
# =============================================
tax["Product_Category"] = tax["Product_Category"].astype(str).str.strip()
tax = tax[(tax["GST"] >= 0) & (tax["GST"] <= 1)]
print(f"Tax clean: {tax.shape}")

Tax clean: (20, 2)


## **2. Merge into Master DataFrame**

Strategy:
- `sales` LEFT JOIN `customers` (add Gender, Location, Tenure)
- LEFT JOIN `tax` (add GST rate)
- LEFT JOIN `discount` on Month + Category (add Discount_pct)
- LEFT JOIN `marketing` on Date (add daily marketing spend)

In [76]:
# Step 1: sales + customers
df = sales.merge(customers, on="CustomerID", how="left")
print(f"After +customers: {df.shape}")

# Step 2: + tax
df = df.merge(tax, on="Product_Category", how="left")
print(f"After +tax: {df.shape}")

# Step 3: + discount (join on Month + Category)
df = df.merge(
    discount[["Month", "Product_Category", "Discount_pct"]],
    on=["Month", "Product_Category"],
    how="left"
)
# Nếu không có discount, fill 0
df["Discount_pct"] = df["Discount_pct"].fillna(0)
# Nếu Coupon_Status != Used thì discount = 0 (coupon có nhưng không dùng)
df.loc[df["Coupon_Status"] != "Used", "Discount_pct"] = 0
print(f"After +discount: {df.shape}")

# Step 4: + marketing (join on date)
marketing_daily = marketing[["Date", "Offline_Spend", "Online_Spend", "Total_Marketing_Spend"]].copy()
marketing_daily = marketing_daily.rename(columns={"Date": "Transaction_Date"})
df = df.merge(marketing_daily, on="Transaction_Date", how="left")
print(f"After +marketing: {df.shape}")

After +customers: (52924, 15)
After +tax: (52924, 16)
After +discount: (52924, 17)
After +marketing: (52924, 20)


In [77]:
df.head()

,CustomerID,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,Revenue,Month,Gender,Location,Tenure_Months,GST,Discount_pct,Offline_Spend,Online_Spend,Total_Marketing_Spend
0,17850,16679,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,153.71,Jan,M,Chicago,12,0.10,0.1,4500,2424.5,6924.5
1,17850,16680,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.5,Used,153.71,Jan,M,Chicago,12,0.10,0.1,4500,2424.5,6924.5
2,17850,16681,2019-01-01,GGOEGFKQ020399,Google Laptop and Cell Phone Stickers,Office,1,2.05,6.5,Used,2.05,Jan,M,Chicago,12,0.10,0.1,4500,2424.5,6924.5
3,17850,16682,2019-01-01,GGOEGAAB010516,Google Men's 100% Cotton Short Sleeve Hero Tee...,Apparel,5,17.53,6.5,Not Used,87.65,Jan,M,Chicago,12,0.18,0.0,4500,2424.5,6924.5
4,17850,16682,2019-01-01,GGOEGBJL013999,Google Canvas Tote Natural/Navy,Bags,1,16.50,6.5,Used,16.50,Jan,M,Chicago,12,0.18,0.1,4500,2424.5,6924.5


In [78]:
# Check null after merging
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print("Null % after merging:")
print(null_pct[null_pct > 0])

Null % after merging:
Series([], dtype: float64)


In [79]:
# Export cleaned data
sales.to_pickle("Dataset/Cleaned_data/clean_sales.pkl")
customers.to_pickle("Dataset/Cleaned_data/clean_customers.pkl")
discount.to_pickle("Dataset/Cleaned_data/clean_discount.pkl")
marketing.to_pickle("Dataset/Cleaned_data/clean_marketing.pkl")
tax.to_pickle("Dataset/Cleaned_data/clean_tax.pkl")
df.to_pickle("Dataset/Cleaned_data/clean_dataset.pkl")

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CustomerID             52924 non-null  str           
 1   Transaction_ID         52924 non-null  str           
 2   Transaction_Date       52924 non-null  datetime64[us]
 3   Product_SKU            52924 non-null  str           
 4   Product_Description    52924 non-null  str           
 5   Product_Category       52924 non-null  str           
 6   Quantity               52924 non-null  int64         
 7   Avg_Price              52924 non-null  float64       
 8   Delivery_Charges       52924 non-null  float64       
 9   Coupon_Status          52924 non-null  str           
 10  Revenue                52924 non-null  float64       
 11  Month                  52924 non-null  str           
 12  Gender                 52924 non-null  str           
 13  Location    